# Phase 8 — Controlled template generation

## Goal

Generate a reviewable internal reference dossier from one signed Phase 7 run.
The supplied Devoteam template is hash-pinned and used only as the visual and
semantic schema. Factual content comes from the canonical Phase 4 catalogue and
exact Phase 7 evidence citations.

This phase is deterministic: no LLM, translation API, source-document copying,
or automatic client delivery is used. Every output remains marked
`DRAFT INTERNE — VALIDATION REQUISE`.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

PROJECT_FOLDER_NAME = "Devoteam_AI_CLEAN_PIPELINE"
PROJECT_PARENT_FOLDER_NAME = "Devoteam internship"
PACKAGE_FILENAME = "PHASE_8_CONTROLLED_TEMPLATE_GENERATION_PACKAGE.zip"
PACKAGE_SHA256 = "bcd591653d16f6f71ccb24564584235a904fe7158de6322d18036935c3e3115b"
PACKAGE_MANIFEST_SHA256 = "8d0311422c7a2a148417265ddb61cf4549c6dbb5b1e26ffdb4bcbf4e1b97e8d1"
SNAPSHOT_ID = "20260714T154731Z_129ff982c8"
PHASE4_RUN_NAME = "phase4_corpus_v1"

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print("Phase 8 contract loaded: deterministic template generation; external calls disabled.")

## 1. Locate the clean project

The Colab path is explicit so Drive discovery cannot stall. Local validation may
set `DEVOTEAM_PROJECT_ROOT` and `DEVOTEAM_PHASE8_PACKAGE`.

In [ ]:
override = os.environ.get("DEVOTEAM_PROJECT_ROOT")
if override:
    PROJECT_ROOT = Path(override).resolve()
else:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = (
        Path("/content/drive/MyDrive")
        / PROJECT_PARENT_FOLDER_NAME
        / PROJECT_FOLDER_NAME
    ).resolve()
    assert PROJECT_ROOT.is_dir(), f"Clean project folder not found: {PROJECT_ROOT}"
    assert (PROJECT_ROOT / "config" / "project.yaml").exists(), "Project configuration is missing"

PACKAGE_PATH = Path(
    os.environ.get("DEVOTEAM_PHASE8_PACKAGE", PROJECT_ROOT / PACKAGE_FILENAME)
).resolve()
TEMPLATE_PATH = Path(
    os.environ.get(
        "DEVOTEAM_PHASE8_TEMPLATE",
        PROJECT_ROOT / "human_inputs" / "phase8" / "REFERENCE_TEMPLATE.docx",
    )
).resolve()
assert PROJECT_ROOT.name == PROJECT_FOLDER_NAME, PROJECT_ROOT
assert PACKAGE_PATH.exists(), f"Missing package: {PACKAGE_PATH}"
assert TEMPLATE_PATH.exists(), f"Missing template: {TEMPLATE_PATH}"
print(f"Project root: {PROJECT_ROOT}")
print(f"Template: {TEMPLATE_PATH}")

## 2. Verify and install the signed Phase 8 overlay

Only manifest-listed files are installed. Identical files are skipped and any
conflict stops the run. The default test is Phase 8 only for speed; set
`DEVOTEAM_RUN_FULL_REGRESSION=1` to rerun every project test.

In [ ]:
assert file_sha256(PACKAGE_PATH) == PACKAGE_SHA256, "Phase 8 package hash mismatch"
with zipfile.ZipFile(PACKAGE_PATH) as archive:
    names = archive.namelist()
    assert "PHASE_8_PACKAGE_MANIFEST.json" in names
    manifest_bytes = archive.read("PHASE_8_PACKAGE_MANIFEST.json")
    assert hashlib.sha256(manifest_bytes).hexdigest() == PACKAGE_MANIFEST_SHA256
    package_manifest = json.loads(manifest_bytes)
    allowed = set(package_manifest["files"]) | {"PHASE_8_PACKAGE_MANIFEST.json"}
    assert set(names) == allowed, "Package contains undeclared files"
    installed = skipped = 0
    for name in names:
        target = (PROJECT_ROOT / name).resolve()
        assert target == PROJECT_ROOT or PROJECT_ROOT in target.parents, name
        data = archive.read(name)
        if name != "PHASE_8_PACKAGE_MANIFEST.json":
            assert hashlib.sha256(data).hexdigest() == package_manifest["files"][name]["sha256"]
        if target.exists():
            assert target.read_bytes() == data, f"Conflicting existing Phase 8 file: {name}"
            skipped += 1
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
            installed += 1

requirements = PROJECT_ROOT / "requirements" / "phase8.txt"
if os.environ.get("DEVOTEAM_SKIP_PIP") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements), "PyMuPDF"], check=True)

environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + environment.get("PYTHONPATH", "")
environment["DEVOTEAM_PHASE8_TEMPLATE"] = str(TEMPLATE_PATH)
test_target = PROJECT_ROOT / ("tests" if os.environ.get("DEVOTEAM_RUN_FULL_REGRESSION") == "1" else "tests/test_phase8_template_generation.py")
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(test_target)],
    cwd=PROJECT_ROOT, env=environment, text=True, capture_output=True,
)
print(tests.stdout[-4000:])
assert tests.returncode == 0, tests.stderr[-4000:]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Package verified: installed={installed}, unchanged={skipped}")

## 3. Resolve and verify the signed inputs

Phase 8 consumes the immutable Phase 4 reference catalogue, exactly one complete
Phase 7 run, and the exact uploaded template. Set
`DEVOTEAM_PHASE7_OPPORTUNITY_ID` only when several recommendation folders exist.

In [ ]:
from devoteam_reference_ai.phase7_recommendations import load_phase7_config
from devoteam_reference_ai.phase8_template_generation import (
    load_phase8_config,
    run_phase8,
    verify_phase8_run,
    verify_reference_template,
)

PHASE7_CONFIG = load_phase7_config(PROJECT_ROOT / "config" / "phase7_recommendations.yaml")
PHASE8_CONFIG = load_phase8_config(PROJECT_ROOT / "config" / "phase8_template_generation.yaml")
PHASE4_ROOT = PROJECT_ROOT / "data" / "canonical" / SNAPSHOT_ID / PHASE4_RUN_NAME
assert PHASE4_ROOT.is_dir(), PHASE4_ROOT

opportunity_id = os.environ.get("DEVOTEAM_PHASE7_OPPORTUNITY_ID", "").strip()
if opportunity_id:
    PHASE7_ROOT = PROJECT_ROOT / "data" / "recommendations" / opportunity_id / PHASE7_CONFIG["pipeline_version"]
    assert PHASE7_ROOT.is_dir(), PHASE7_ROOT
else:
    candidates = sorted(
        path for path in (PROJECT_ROOT / "data" / "recommendations").glob(f"*/{PHASE7_CONFIG['pipeline_version']}")
        if (path / PHASE7_CONFIG["output"]["success_marker"]).exists()
    )
    assert len(candidates) == 1, (
        "Expected one complete Phase 7 run; set DEVOTEAM_PHASE7_OPPORTUNITY_ID. "
        f"Found: {candidates}"
    )
    PHASE7_ROOT = candidates[0]

template_check = verify_reference_template(TEMPLATE_PATH, PHASE8_CONFIG)
print(f"Phase 4: {PHASE4_ROOT}")
print(f"Phase 7: {PHASE7_ROOT}")
print(f"Template SHA-256: {template_check['sha256']}")
print(f"Template reference slots: {template_check['reference_slots']}")

## 4. Generate, validate, and sign the dossier

For the packaged synthetic sample, the top five technical rows are used only as
a development fixture. A real opportunity must first have a complete Phase 7
business review with explicit `SHORTLIST`/`REJECT` decisions.

In [ ]:
RUN_ROOT, MANIFEST = run_phase8(
    phase4_root=PHASE4_ROOT,
    phase7_root=PHASE7_ROOT,
    template_path=TEMPLATE_PATH,
    phase7_config=PHASE7_CONFIG,
    phase8_config=PHASE8_CONFIG,
    output_root=PROJECT_ROOT / PHASE8_CONFIG["output"]["root"],
)
verification = verify_phase8_run(RUN_ROOT, PHASE8_CONFIG)
assert verification["manifest"] == MANIFEST

print("PHASE 8 CONTROLLED TEMPLATE GENERATION: PASS")
print(f"Status: {MANIFEST['status']}")
print(f"Selected references: {MANIFEST['selected_references']}")
print(f"Citation rows: {MANIFEST['citation_rows']}")
print(f"Citation coverage: {MANIFEST['citation_coverage']:.0%}")
print(f"References requiring evidence review: {MANIFEST['references_requiring_evidence_review']}")
print("External LLM / translation calls: 0 / 0")
print(f"DOCX: {RUN_ROOT / PHASE8_CONFIG['output']['docx_name']}")
print(f"PDF:  {RUN_ROOT / PHASE8_CONFIG['output']['pdf_name']}")
print(f"Output: {RUN_ROOT}")
if "SAMPLE_ONLY" in MANIFEST["status"]:
    print("IMPORTANT: synthetic technical sample only; do not send this dossier to a client.")
if MANIFEST["references_requiring_evidence_review"]:
    print("BLOCKER: review the references explicitly marked 'À revoir' before any external use.")

## Next step

Open `REFERENCE_DOSSIER_DRAFT.docx` from the printed output folder. Review the
wording, evidence relevance, confidentiality, and commercial suitability with
Devoteam business and security owners. The current sample contains two explicit
evidence-review warnings and is not a client deliverable. Production promotion
also remains dependent on the independent Phase 5.1 expert evaluation gate.